# Model Experiment: XGBoost

This notebook continues the feature engineering organized into two reusable classes: `WalmartBasePreprocessor` (merge, missing values,calendar/holiday features) and `WalmartTabularFeatureEngineer` (tree-model-specific: typeencoding, lag/rolling sales history).

Both classes are leakage-safe: lag/rolling features for validation or test data are computed from history seen during `fit()`, never from that data's own sales values.

In [11]:
import mlflow
import xgboost as xgb
import numpy as np
import pandas as pd
import optuna
import dagshub
import sys

sys.path.append("..")

from src.data.load_data import load_raw_data
from src.data.splits import last_n_weeks_split
from src.features.base import WalmartBasePreprocessor
from src.features.tabular import WalmartTabularFeatureEngineer

dagshub.init(repo_owner="LukaBatilashvili07", repo_name="walmart-sales-forecasting", mlflow=True)
mlflow.set_experiment("XGBoost_Training")
mlflow.xgboost.autolog(log_models=False)

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

## 1. Data Loading and Train/Validation Split

A time-based split is required here instead of a random split, because random splitting would let the model train on data
from after some validation dates, a leakage problem specific to time series.

In [12]:
train_raw, test_raw, stores, features = load_raw_data("../data/raw")

train_part, valid_part = last_n_weeks_split(train_raw, n_weeks=39)

print(train_part["Date"].min(), "-", train_part["Date"].max())
print(valid_part["Date"].min(), "-", valid_part["Date"].max())

2010-02-05 00:00:00 - 2012-01-27 00:00:00
2012-02-03 00:00:00 - 2012-10-26 00:00:00


## 2. XGBoost_Cleaning

WalmartBasePreprocessor is fit on stores and features then applied separately to train_part and valid_part. This confirms the merge and missing-value handling behave consistently on both splits before any model-specific feature engineering happens.

In [13]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    base_pre = WalmartBasePreprocessor()
    base_pre.fit(stores, features)

    train_base = base_pre.transform(train_part)
    valid_base = base_pre.transform(valid_part)

    mlflow.log_param("markdown_start_date", str(base_pre.MARKDOWN_START_DATE.date()))
    mlflow.log_metric("train_rows", len(train_base))
    mlflow.log_metric("valid_rows", len(valid_base))
    mlflow.log_metric("train_nan_total", int(train_base.isna().sum().sum()))
    mlflow.log_metric("valid_nan_total", int(valid_base.isna().sum().sum()))

    train_base.to_parquet("_cache_train_base.parquet")
    valid_base.to_parquet("_cache_valid_base.parquet")
    mlflow.log_artifact("_cache_train_base.parquet")
    mlflow.log_artifact("_cache_valid_base.parquet")

🏃 View run XGBoost_Cleaning at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/e259e9491597406aa10d0a92c7c061bd
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


## 3. Run: XGBoost_Feature_Selection

Tabular features are added on top of base preprocessing. `fit()` uses only train_part own sales history, `transform_train()` runs on the training data, and `transform_future()` runs on validation, so validation sales never leak into lag or rolling features.

The feature list covers five groups: store/department identity, calendar and holiday flags, markdown aggregates, economic indicators, and sales history.

In [14]:
train_base = pd.read_parquet("_cache_train_base.parquet")
valid_base = pd.read_parquet("_cache_valid_base.parquet")

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    tab_eng = WalmartTabularFeatureEngineer()
    tab_eng.fit(train_base[["Store", "Dept", "Date", "Weekly_Sales"]])

    train_full = tab_eng.transform_train(train_base)
    valid_full = tab_eng.transform_future(valid_base)

    feature_cols = [
        "Store", "Dept", "Size", "Temperature", "Fuel_Price",
        "CPI", "Unemployment",
        "Type_encoded",
        "Year", "Month", "WeekOfYear", "Week_sin", "Week_cos",
        "IsSuperBowl", "IsLaborDay", "IsThanksgiving", "IsChristmas",
        "total_markdown", "abs_total_markdown",
        "positive_markdown_sum", "negative_markdown_sum",
        "has_markdown_signal", "markdown_missing_count", "markdown_available_period",
        "Sales_lag_1", "Sales_lag_4", "Sales_lag_52",
        "Sales_roll_mean_4", "Sales_roll_std_4", "Sales_roll_mean_12",
    ]

    X_train, y_train = train_full[feature_cols], train_full["Weekly_Sales"]
    X_valid, y_valid = valid_full[feature_cols], valid_full["Weekly_Sales"]

    baseline = xgb.XGBRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
    baseline.fit(X_train, y_train)

    importances = pd.Series(baseline.feature_importances_, index=feature_cols).sort_values(ascending=False)
    preds = baseline.predict(X_valid)
    baseline_wmae = wmae(y_valid.values, preds, valid_full["IsHoliday"].values)

    mlflow.log_param("n_features", len(feature_cols))
    mlflow.log_metric("baseline_wmae", baseline_wmae)
    importances.to_frame("importance").to_csv("_feature_importance.csv")
    mlflow.log_artifact("_feature_importance.csv")

    print(importances.head(15))
    print("Baseline WMAE:", baseline_wmae)

    train_full.to_parquet("_cache_train_full.parquet")
    valid_full.to_parquet("_cache_valid_full.parquet")

2026/07/11 12:40:56 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Sales_lag_1               0.593806
Sales_roll_mean_4         0.245583
IsThanksgiving            0.025722
WeekOfYear                0.018193
Sales_lag_52              0.017207
Sales_lag_4               0.012967
Year                      0.011172
Sales_roll_std_4          0.008983
has_markdown_signal       0.007905
Week_sin                  0.006309
IsSuperBowl               0.006133
Dept                      0.005829
Month                     0.005684
markdown_missing_count    0.005056
total_markdown            0.004079
dtype: float32
Baseline WMAE: 6512.076963308009
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/5286f7579af2445fa65d41446f518582
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


## 4. Run: XGBoost_CV Walk-Forward Cross-Validation

Regular k-fold CV is unsafe here, since random folds would let the model train on rows from after some validation dates. Instead, we use walk-forward folds: each training window grows forward in time, and its matching validation fold always comes right after it.

In [15]:
train_base = pd.read_parquet("_cache_train_base.parquet")

def make_walk_forward_folds(dates, n_folds=3, val_weeks=13):
    dates = sorted(dates)
    folds = []
    for i in range(n_folds, 0, -1):
        cutoff = len(dates) - i * val_weeks
        if cutoff <= 0:
            continue
        train_dates = dates[:cutoff]
        val_dates = dates[cutoff:cutoff + val_weeks]
        folds.append((train_dates, val_dates))
    return folds

unique_dates = train_base["Date"].unique()
folds = make_walk_forward_folds(unique_dates, n_folds=3, val_weeks=13)

with mlflow.start_run(run_name="XGBoost_CV"):
    fold_scores = []

    for i, (train_dates, val_dates) in enumerate(folds):
        with mlflow.start_run(run_name=f"XGBoost_CV_fold_{i}", nested=True):
            fold_train_raw = train_base[train_base["Date"].isin(train_dates)]
            fold_val_raw = train_base[train_base["Date"].isin(val_dates)]

            fold_tab_eng = WalmartTabularFeatureEngineer()
            fold_tab_eng.fit(fold_train_raw[["Store", "Dept", "Date", "Weekly_Sales"]])

            fold_train_full = fold_tab_eng.transform_train(fold_train_raw)
            fold_val_full = fold_tab_eng.transform_future(fold_val_raw)

            X_tr = fold_train_full[feature_cols]
            y_tr = fold_train_full["Weekly_Sales"]
            X_val = fold_val_full[feature_cols]
            y_val = fold_val_full["Weekly_Sales"]

            model = xgb.XGBRegressor(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
            model.fit(X_tr, y_tr)
            preds = model.predict(X_val)
            score = wmae(y_val.values, preds, fold_val_full["IsHoliday"].values)

            fold_scores.append(score)
            mlflow.log_metric("fold_wmae", score)
            print(f"Fold {i}: train up to {train_dates[-1]}, val {val_dates[0]}-{val_dates[-1]}, WMAE={score:.2f}")

    mlflow.log_metric("cv_wmae_mean", float(np.mean(fold_scores)))
    mlflow.log_metric("cv_wmae_std", float(np.std(fold_scores)))
    print("CV WMAE mean/std:", np.mean(fold_scores), np.std(fold_scores))

2026/07/11 13:02:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fold 0: train up to 2011-04-29 00:00:00, val 2011-05-06 00:00:00-2011-07-29 00:00:00, WMAE=6660.82
🏃 View run XGBoost_CV_fold_0 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/89bf2db94d604d938f2a2fdb0662e16b
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


2026/07/11 13:02:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fold 1: train up to 2011-07-29 00:00:00, val 2011-08-05 00:00:00-2011-10-28 00:00:00, WMAE=7356.73
🏃 View run XGBoost_CV_fold_1 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/018d276c8f814cf1965318e6dc25048e
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


2026/07/11 13:02:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


Fold 2: train up to 2011-10-28 00:00:00, val 2011-11-04 00:00:00-2012-01-27 00:00:00, WMAE=7966.82
🏃 View run XGBoost_CV_fold_2 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/4aaa48fd9fdf4c369da23368c7063a4f
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2
CV WMAE mean/std: 7328.12098936518 533.5566327628782
🏃 View run XGBoost_CV at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/302606fd2d0d4431b9f7818f833cc0ba
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


If fold scores are close together, the model behaves consistently over time, and the single 39-week validation score can be trusted. If one fold is much worse (for example Christmas), that is a real limitation worth naming in the report.

## 5. Run: XGBoost_HPO

Optuna searches for hyperparameters that directly minimize WMAE the competition's own metric, where holiday weeks count five times more than normal weeks. This matches how the model will actually be judged, rather than optimizing a generic metric like RMSE. 

In [16]:
train_full = pd.read_parquet("_cache_train_full.parquet")
valid_full = pd.read_parquet("_cache_valid_full.parquet")

X_train, y_train = train_full[feature_cols], train_full["Weekly_Sales"]
X_valid, y_valid = valid_full[feature_cols], valid_full["Weekly_Sales"]
holiday_valid = valid_full["IsHoliday"].values

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 5.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

    with mlflow.start_run(run_name=f"XGBoost_HPO_trial_{trial.number}", nested=True):
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_valid)
        score = wmae(y_valid.values, preds, holiday_valid)
        mlflow.log_metric("wmae", score)

    return score

with mlflow.start_run(run_name="XGBoost_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=30)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_wmae", study.best_value)

best_params = study.best_params
print("Best parameters:", best_params)
print("Best WMAE:", study.best_value)

[I 2026-07-11 13:38:48,456] A new study created in memory with name: no-name-89729526-29f9-4c41-b16e-40e6e2b9727e
2026/07/11 13:38:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


🏃 View run XGBoost_HPO_trial_0 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/7745e897a7c94ddeacd9aeff6870935f
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


[I 2026-07-11 13:38:59,521] Trial 0 finished with value: 5025.353828966972 and parameters: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.046153131923979705, 'subsample': 0.9628891233628044, 'colsample_bytree': 0.7255562482158516, 'min_child_weight': 9, 'reg_lambda': 0.3208675859542617}. Best is trial 0 with value: 5025.353828966972.
2026/07/11 13:39:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns

🏃 View run XGBoost_HPO_trial_1 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/5f4783129a00409fbe6bcb7d1ec6b64b
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


[I 2026-07-11 13:39:12,018] Trial 1 finished with value: 4734.534412312879 and parameters: {'n_estimators': 800, 'max_depth': 8, 'learning_rate': 0.16235523081328448, 'subsample': 0.63300766750466, 'colsample_bytree': 0.7125134213926402, 'min_child_weight': 10, 'reg_lambda': 4.370372062288559}. Best is trial 1 with value: 4734.534412312879.
2026/07/11 13:39:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as 

...

🏃 View run XGBoost_HPO_trial_28 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/6a81c2cc67e34d5f82fae715c676e8fd
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


[I 2026-07-11 13:46:04,217] Trial 28 finished with value: 5337.202471068199 and parameters: {'n_estimators': 200, 'max_depth': 8, 'learning_rate': 0.023819763256103778, 'subsample': 0.6945788734179721, 'colsample_bytree': 0.7533683198670019, 'min_child_weight': 6, 'reg_lambda': 0.7857252934786454}. Best is trial 22 with value: 4638.494813585583.
2026/07/11 13:46:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer column

🏃 View run XGBoost_HPO_trial_29 at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/d26103304d0040fb857ff1376a28e4dd
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


[I 2026-07-11 13:46:13,474] Trial 29 finished with value: 5120.495384774613 and parameters: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.05041446664114512, 'subsample': 0.7309297176784458, 'colsample_bytree': 0.8696274178110243, 'min_child_weight': 3, 'reg_lambda': 0.5117759713274975}. Best is trial 22 with value: 4638.494813585583.


🏃 View run XGBoost_HPO at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/5eedec221b2d4009bf30cd0534c8a1cf
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2
Best parameters: {'n_estimators': 800, 'max_depth': 7, 'learning_rate': 0.03527265332711685, 'subsample': 0.8044283240415013, 'colsample_bytree': 0.8755310106592203, 'min_child_weight': 3, 'reg_lambda': 1.5671876931621156}
Best WMAE: 4638.494813585583


## 6. Run: XGBoost_Best

Best model of architecture saved as a pipeline.

`WalmartXGBPipeline` wraps the fitted `base_preprocessor`, the fitted `tabular_engineer`,
and the trained XGBoost model into one object. Its `predict()` runs all three steps in
order. It is logged with `mlflow.pyfunc.log_model()`, MLflow's standard way to package a
custom, non-scikit-learn preprocessing chain as one deployable model.

In [17]:
class WalmartXGBPipeline(mlflow.pyfunc.PythonModel):
    def __init__(self, base_preprocessor, tabular_engineer, model, feature_cols):
        self.base_preprocessor = base_preprocessor
        self.tabular_engineer = tabular_engineer
        self.model = model
        self.feature_cols = feature_cols

    def predict(self, context, model_input: pd.DataFrame):
        base = self.base_preprocessor.transform(model_input)
        full = self.tabular_engineer.transform_future(base)
        X = full[self.feature_cols]
        return self.model.predict(X)


with mlflow.start_run(run_name="XGBoost_Best") as run:
    final_model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model.fit(X_train, y_train)

    preds = final_model.predict(X_valid)
    final_wmae = wmae(y_valid.values, preds, holiday_valid)
    mlflow.log_params(best_params)
    mlflow.log_metric("final_valid_wmae", final_wmae)
    print("Final validation WMAE:", final_wmae)

    pipeline = WalmartXGBPipeline(
        base_preprocessor=base_pre,
        tabular_engineer=tab_eng,
        model=final_model,
        feature_cols=feature_cols,
    )

    mlflow.pyfunc.log_model(
        artifact_path="xgboost_pipeline",
        python_model=pipeline,
        registered_model_name="Walmart_XGBoost_Pipeline",
    )

2026/07/11 14:31:02 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/07/11 14:31:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Final validation WMAE: 4638.494813585583


2026/07/11 14:31:05 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/11 14:31:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Walmart_XGBoost_Pipeline'.
2026/07/11 14:31:20 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_XGBoost_Pipeline, version 1
Created version '1' of model 'Walmart_XGBoost_Pipeline'.


🏃 View run XGBoost_Best at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/5c27f8614fa047dab507b30290f86c3e
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2


### Verifying the Pipeline works on raw data

This is the same check `model_inference.ipynb` will perform later.

In [21]:
loaded = mlflow.pyfunc.load_model("models:/Walmart_XGBoost_Pipeline/1")
test_preds = loaded.predict(test_raw)
print(test_preds[:10])

2026/07/11 19:09:27 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."


[22391.877 28627.852 33943.305 24898.172 27698.49  27144.781 22763.367
 26978.734 20340.035 23174.047]


## 7. Run: XGBoost_Final_Refit

Refit the best-params model on the ENTIRE train.csv (all 143 weeks), not just train_part (104 weeks)

In [22]:
with mlflow.start_run(run_name="XGBoost_Final_Refit") as run:
    full_base_pre = WalmartBasePreprocessor()
    full_base_pre.fit(stores, features)
    train_full_all = full_base_pre.transform(train_raw)

    full_tab_eng = WalmartTabularFeatureEngineer()
    full_tab_eng.fit(train_full_all[["Store", "Dept", "Date", "Weekly_Sales"]])
    train_full_all = full_tab_eng.transform_train(train_full_all)

    X_train_all = train_full_all[feature_cols]
    y_train_all = train_full_all["Weekly_Sales"]

    final_model_full = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    final_model_full.fit(X_train_all, y_train_all)

    mlflow.log_params(best_params)
    mlflow.log_param("training_data", "full_train_143_weeks")
    mlflow.log_metric("train_rows", len(train_full_all))

    final_pipeline = WalmartXGBPipeline(
        base_preprocessor=full_base_pre,
        tabular_engineer=full_tab_eng,
        model=final_model_full,
        feature_cols=feature_cols,
    )

    mlflow.pyfunc.log_model(
        artifact_path="xgboost_pipeline_full",
        python_model=final_pipeline,
        registered_model_name="Walmart_XGBoost_Pipeline",
    )

    print("Final model trained on full train.csv:", len(train_full_all), "rows")

2026/07/12 16:54:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 16:54:26 WARNING mlflow.pyfunc: Failed to infer model signature: Type hint <input: <class 'pandas.core.frame.DataFrame'>, output: None> cannot be used to infer model signature and input example is not provided, model signature cannot be inferred.
2026/07/12 16:54:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'Walmart_XGBoost_Pipeline' already exists. Creating a new version of this model...
2026/07/12 16:54:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_XGBoost_Pipeline, version 2
Created version '2' of model 'Walmart_XGBoost_Pipeline'.


Final model trained on full train.csv: 421570 rows
🏃 View run XGBoost_Final_Refit at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2/runs/21b6aa9b033b4041803ba59797652bb6
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/2
